# 20 — Ablation Study: Mutation Operators
For each of the 10 mutation operators, measure accuracy contribution on GSM8K 50-subset,
frequency selected during beam search, and rank among all operators.
Output: bar chart of mutation effectiveness + ranked table.

In [ ]:
import sys, json, copy
from pathlib import Path
from collections import defaultdict

REPO_ROOT = Path("../..")
sys.path.insert(0, str(REPO_ROOT / "src"))

from evaluation.benchmarks import BenchmarkLoader
from evaluation.metrics import EvaluationMetrics
from llm.cache import ResponseCache
from prompts.template import PromptTemplate
from prompts.mutations import PromptMutator
from search.beam_search import BeamSearchPromptOptimizer

cache = ResponseCache(db_path=str(REPO_ROOT / "data" / "cache" / "responses.db"))
loader = BenchmarkLoader(cache_dir=REPO_ROOT / "data" / "benchmarks")

In [ ]:
gsm8k = loader.load_gsm8k(subset_size=500)
subset = gsm8k[:50]
val_set = [{"input": ex["question"], "expected": ex["answer"]} for ex in subset]

In [ ]:
from llm.base import BaseLLMClient

class CachedMockLLM(BaseLLMClient):
    def complete(self, prompt, **kwargs):
        cached = cache.get(prompt, model="claude-3-haiku", temperature=0.0)
        if cached:
            return cached, {"input_tokens": 0, "output_tokens": 0}
        response = "[mock]"
        cache.set(prompt, model="claude-3-haiku", temperature=0.0, response=response)
        return response, {"input_tokens": len(prompt.split()), "output_tokens": 5}

llm = CachedMockLLM()

## Isolate Each Mutation Operator
Run beam search restricted to a single mutation at a time and record accuracy.

In [ ]:
mutator = PromptMutator()
all_mutations = mutator.all_mutations(PromptTemplate(task="Answer the following question."))
mutation_names = [m.mutation_path()[-1] if m.mutation_path() else f"mutation_{i}" for i, m in enumerate(all_mutations)]
print(f"{len(all_mutations)} mutation operators:", mutation_names)

In [ ]:
ground_truth = [ex["answer"] for ex in subset]

def eval_single_mutation(mutated_template):
    predictions = []
    for ex in subset:
        prompt = mutated_template.render(question=ex["question"])
        resp, _ = llm.complete(prompt)
        predictions.append(resp)
    return EvaluationMetrics.accuracy(predictions, ground_truth)

base_template = PromptTemplate(task="Answer the following question.")
base_acc = eval_single_mutation(base_template)
print(f"Baseline accuracy: {base_acc:.4f}")

ablation_results = []
for name, m in zip(mutation_names, all_mutations):
    acc = eval_single_mutation(m)
    ablation_results.append({"mutation": name, "accuracy": round(acc, 4),
                              "delta": round(acc - base_acc, 4)})
    print(f"  {name}: {acc:.4f} (delta={acc - base_acc:+.4f})")

## Selection Frequency During Full Beam Search

In [ ]:
optimizer = BeamSearchPromptOptimizer(beam_width=3, max_iterations=5, llm_client=llm, mutator=mutator)
optimizer.search(base_template, val_set)

freq_map = defaultdict(int)
for entry in optimizer.history:
    path = entry.get("best_path", "")
    for name in mutation_names:
        if name in path:
            freq_map[name] += 1

for r in ablation_results:
    r["frequency"] = freq_map.get(r["mutation"], 0)

ablation_results.sort(key=lambda x: x["accuracy"], reverse=True)
for rank, r in enumerate(ablation_results, 1):
    r["rank"] = rank

## Results Table

In [ ]:
print(f"{'Rank':<5} {'Mutation':<30} {'Accuracy':<10} {'Delta':<10} {'Frequency':<10}")
print("-" * 65)
for r in ablation_results:
    print(f"{r['rank']:<5} {r['mutation']:<30} {r['accuracy']:<10} {r['delta']:<10} {r['frequency']:<10}")

## Bar Chart

In [ ]:
import matplotlib.pyplot as plt

names = [r["mutation"] for r in ablation_results]
accs  = [r["accuracy"] for r in ablation_results]
colors = ["steelblue" if r["delta"] >= 0 else "tomato" for r in ablation_results]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(names[::-1], accs[::-1], color=colors[::-1])
ax.axvline(base_acc, color="black", linestyle="--", label=f"Baseline ({base_acc:.3f})")
ax.set_xlabel("Accuracy on GSM8K 50-subset")
ax.set_title("Mutation Operator Ablation Study")
ax.legend()
plt.tight_layout()
plt.savefig(REPO_ROOT / "data" / "results" / "ablation_chart.png", dpi=150)
plt.show()
print("Chart saved to data/results/ablation_chart.png")

## Conclusion

In [ ]:
top3 = ablation_results[:3]
print("Top 3 mutations to prioritize for domain-specific tasks:")
for r in top3:
    print(f"  #{r['rank']} {r['mutation']} — accuracy {r['accuracy']:.4f}, selected {r['frequency']}x in beam search")